# MODELO LSTM PARA EL CONTAMINANTE CO PARA MADRID

En este notebook, vamos a ajustar modelos LSTM para el contaminante CO sobre la ciudad de Madrid.

Los modelos de memoria a corto y largo plazo (LSTM, por sus siglas en inglés *Long Short-Term Memory*) son una variante de las redes neuronales recurrentes (RNN) que están especialmente diseñadas para el análisis de series temporales. A diferencia de las RNN tradicionales, las LSTM pueden capturar y aprender de dependencias a largo plazo en datos secuenciales. Esto las hace especialmente útiles para predecir valores futuros en series temporales complejas, donde las relaciones y los patrones pueden extenderse a lo largo de períodos prolongados.

Los modelos LSTM utilizan unidades de memoria llamadas "Memoria LSTM", las cuales pueden recordar u olvidar información relevante a medida que procesan datos secuenciales. Estas celdas permiten mantener una memoria a largo plazo de las dependencias encontradas en la serie temporal, otorgándoles una capacidad única para capturar patrones complejos y realizar predicciones precisas.

Importamos las librerías y definimos las rutas.

In [56]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import random
import warnings

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error


from pylab import rcParams
plt.style.use("fivethirtyeight")
plt.rcParams["lines.linewidth"] = 1.5
light_style = {
    "figure.facecolor": "#d9effb",   
    "axes.facecolor": "#d9effb",
    "savefig.facecolor": "#d9effb",
    "axes.grid": True,
    "axes.grid.which": "both",
    "axes.spines.left": True,
    "axes.spines.right": True,
    "axes.spines.top": True,
    "axes.spines.bottom": True,
    "grid.color": "#a9d3f2",
    "grid.linewidth": "0.8",
    "text.color": "#333333",
    "axes.labelcolor": "#333333",
    "axes.labelweight": "black",      
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "font.size": 12,
    "axes.titleweight": "bold",       
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
}
plt.rcParams.update(light_style)
rcParams["figure.figsize"] = (18, 7)

import sys
import importlib
from pathlib import Path

SCRIPTS_PATH = Path.cwd().parents[2]

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.append(str(SCRIPTS_PATH))

import utils
importlib.reload(utils)
from utils import EVALUAR_METRICAS

In [57]:
BASE_PATH = Path("..", "..", "..", "..")
FOLDER_DATA = BASE_PATH / "datasets" / "eda_archivos_cont_clima_indices"

## Carga de los datos y división del conjunto de datos

Cargamos los datos.

In [58]:
df = pd.read_csv(FOLDER_DATA / "dataset_cont_clima_indices_limpio.csv")

Filtramos las columnas que realmente necesitamos y como ciudad elegimos únicamente Madrid. Dado que todo el trabajo exploratorio ya se encuentra realizado en el *notebook* para el modelo Prophet, nos quedamos únicamente con las variables exógenas definidas anteriormente.

In [59]:
# Listado de columnas seleccionadas
columnas = [
    'Start', 'CO (mg.m-3)', 'city', 'temperature_2m', 'snowfall',
    'relative_humidity_2m', 'precipitation', 'rain',
    'surface_pressure', 'cloudcover', 'windspeed_10m', 
    'shortwave_radiation', 'boundary_layer_height', 'NDVI', 'NDBI', 'Año'
]

# Filtrar por Madrid y seleccionar las columnas
df1 = df[df['city'] == 'Madrid'][columnas].copy()

# Resetear los índices para que empiecen desde 0
df1.reset_index(drop=True, inplace=True)

# Eliminamos la columna 'city'
df1.drop(columns=['city'], inplace=True)

In [60]:
# ==============================================================================
# División cronológica del conjunto de datos
# ==============================================================================

train = df1[df1["Año"] <= 2020].copy()

validation = df1[
    (df1["Año"] >= 2021) &
    (df1["Año"] <= 2022)
].copy()

test = df1[df1["Año"] >= 2023].copy()

print(f"Entrenamiento: {train['Start'].min()} -> {train['Start'].max()}")
print(f"Validación:    {validation['Start'].min()} -> {validation['Start'].max()}")
print(f"Prueba:        {test['Start'].min()} -> {test['Start'].max()}")

print()
print(f"Nº muestras entrenamiento: {len(train):,}")
print(f"Nº muestras validación:    {len(validation):,}")
print(f"Nº muestras prueba:        {len(test):,}")

Entrenamiento: 2013-01-01 00:00:00 -> 2020-12-31 23:00:00
Validación:    2021-01-01 00:00:00 -> 2022-12-31 23:00:00
Prueba:        2023-01-01 00:00:00 -> 2024-12-31 23:00:00

Nº muestras entrenamiento: 70,128
Nº muestras validación:    17,520
Nº muestras prueba:        17,544


Para seguir con la forma en que Prophet denominaba a la variable objetivo y la columna temporal, renombramos las fechas por *ds* t la variable objetivo por *y*.

In [61]:
train = train.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
validation = validation.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
test = test.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})

In [62]:
variables_exogenas= [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "cloudcover",
    "windspeed_10m",
    "shortwave_radiation",
    "boundary_layer_height",
    "NDVI",
    "NDBI"
]

Nos quedamos exclusivamente con las variables necesarias para el modelado.

In [63]:
# Lista de columnas que quieres mantener
columnas = ['ds', 'y'] + variables_exogenas

train = train[columnas]
validation = validation[columnas]
test = test[columnas]

## Escalado y creación de las secuencias temporales

### Escalado de los datos

Antes de construir las secuencias temporales, normalizamos tanto la variable objetivo como las variables exógenas. En este caso, utilizaremos dos escaladores independientes: uno para la variable objetivo y otro para las variables exógenas. Es importante que los escaladores sean ajustados exclusivamente sobre el conjunto de entrenamiento, evitando de esta forma introducir información procedente de los conjuntos de validación o prueba.

In [64]:
# ==============================================================================
# Copias de los conjuntos de datos
# ==============================================================================

train_lstm = train.copy()
validation_lstm = validation.copy()
test_lstm = test.copy()


# ==============================================================================
# Conversión y ordenación de las fechas
# ==============================================================================

for df_lstm in [train_lstm, validation_lstm, test_lstm]:
    df_lstm["ds"] = pd.to_datetime(df_lstm["ds"])
    df_lstm.sort_values("ds", inplace=True)
    df_lstm.reset_index(drop=True, inplace=True)

In [65]:
# ==============================================================================
# Escaladores
# ==============================================================================

scaler_y = MinMaxScaler(feature_range=(0, 1))
scaler_X = MinMaxScaler(feature_range=(0, 1))


# ==============================================================================
# Ajuste de los escaladores ÚNICAMENTE sobre entrenamiento
# ==============================================================================

scaler_y.fit(train_lstm[["y"]])
scaler_X.fit(train_lstm[variables_exogenas])


# ==============================================================================
# Transformación de los tres conjuntos
# ==============================================================================

for df_lstm in [train_lstm, validation_lstm, test_lstm]:

    df_lstm["y"] = scaler_y.transform(
        df_lstm[["y"]]
    ).ravel()

    df_lstm[variables_exogenas] = scaler_X.transform(
        df_lstm[variables_exogenas]
    )

### Creación de las secuencias temporales

A diferencia de modelos como Prophet, las redes LSTM necesitan recibir la información organizada en secuencias. Para ello, se construyen ventanas temporales de longitud `input_size`, utilizando los valores históricos de la variable objetivo y de las variables exógenas para predecir el valor de CO correspondiente al siguiente instante temporal.

Por ejemplo, si `input_size = 24`, cada observación utilizará las 24 horas anteriores para estimar la concentración de CO de la hora siguiente.

In [66]:
def crear_dataset_lstm(df, input_size, variables_exogenas):
    """
    Crea secuencias temporales para entrenar una red LSTM.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con la variable objetivo 'y' y las variables exógenas.

    input_size : int
        Número de instantes temporales anteriores utilizados
        para predecir el siguiente valor.

    variables_exogenas : list
        Lista de variables exógenas utilizadas como predictores.

    Retorna
    -------
    X : np.ndarray
        Matriz tridimensional con forma:
        (n_muestras, input_size, n_variables).

    y : np.ndarray
        Valores de la variable objetivo correspondientes
        al instante inmediatamente posterior a cada secuencia.
    """

    # Variables utilizadas como entrada
    columnas_entrada = ["y"] + variables_exogenas

    valores_X = df[columnas_entrada].values.astype(np.float32)
    valores_y = df["y"].values.astype(np.float32)

    X = []
    y = []

    for i in range(input_size, len(df)):

        # Ventana con los input_size instantes anteriores
        X.append(
            valores_X[i - input_size:i]
        )

        # Valor de y en el instante siguiente
        y.append(
            valores_y[i]
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32)
    )

In [67]:
def crear_dataset_lstm_validacion(
    train,
    validation,
    input_size,
    variables_exogenas
):
    """
    Crea las secuencias de validación incorporando las últimas
    observaciones del conjunto de entrenamiento como contexto.
    """

    # Últimas observaciones del entrenamiento necesarias
    contexto = train.tail(input_size)

    # Las concatenamos con validación
    df_completo = pd.concat(
        [contexto, validation],
        axis=0,
        ignore_index=True
    )

    columnas_entrada = ["y"] + variables_exogenas

    valores_X = df_completo[
        columnas_entrada
    ].values.astype(np.float32)

    valores_y = df_completo[
        "y"
    ].values.astype(np.float32)

    X = []
    y = []

    # Comenzamos justo donde empieza validación
    for i in range(input_size, len(df_completo)):

        X.append(
            valores_X[i - input_size:i]
        )

        y.append(
            valores_y[i]
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32)
    )

## Selección de los mejores hiperparámetros

Una vez preparados y escalados los conjuntos de entrenamiento, validación y prueba, continuamos con la selección de la configuración más adecuada de la red neuronal **LSTM**. Para ello, cada modelo se ajusta utilizando exclusivamente las observaciones del conjunto de entrenamiento, correspondientes al periodo comprendido hasta el año 2020, y su capacidad predictiva se evalúa posteriormente sobre el conjunto de validación, formado por los años 2021 y 2022.

Al tratarse de una serie temporal, se mantiene en todo momento el orden cronológico de las observaciones, evitando realizar particiones aleatorias o procedimientos de validación cruzada convencional que pudieran provocar fugas de información entre distintos periodos temporales. Del mismo modo, durante el entrenamiento de las redes se establece `shuffle = False`, de forma que las secuencias mantienen el orden temporal con el que fueron construidas.

La búsqueda de hiperparámetros se realiza mediante `ParameterGrid`, generando todas las combinaciones posibles a partir de los valores definidos previamente. Para cada configuración se construyen las secuencias temporales correspondientes, se entrena una nueva red LSTM sobre el conjunto de entrenamiento y se generan predicciones sobre el conjunto de validación. Posteriormente, las predicciones se transforman nuevamente a la escala original de la concentración de CO y se calcula el error cuadrático medio, MSE, entre los valores reales y estimados. La configuración seleccionada será aquella que presente el menor MSE sobre el conjunto de validación.

A diferencia de Prophet y NeuralProphet, las redes LSTM no reciben directamente cada observación de la serie de manera independiente, sino que necesitan que los datos se organicen en **secuencias temporales**. Para ello, se utiliza el hiperparámetro `input_size`, que determina el número de observaciones anteriores empleadas para predecir la concentración de CO correspondiente al siguiente instante temporal.

Dado que los datos presentan una frecuencia horaria, se consideran ventanas de 24, 48, 72 y 168 observaciones. Estas configuraciones permiten proporcionar al modelo información correspondiente, respectivamente, al último día, los dos últimos días, los tres últimos días o la última semana completa. En todos los casos, el modelo realiza una predicción a una hora vista.

Cada secuencia de entrada contiene tanto los valores históricos de la propia concentración de CO como las variables exógenas seleccionadas previamente. Por tanto, para cada uno de los instantes incluidos en la ventana temporal se utilizan simultáneamente la variable objetivo y las diez variables ambientales y meteorológicas consideradas en el estudio.

Tras el análisis previo de multicolinealidad, se excluyen `rain` y `snowfall` debido a su elevada redundancia con la precipitación total. Las variables exógenas incorporadas finalmente al modelo son:

* Temperatura a dos metros.
* Humedad relativa a dos metros.
* Precipitación total.
* Presión en superficie.
* Nubosidad.
* Velocidad del viento a diez metros.
* Radiación solar de onda corta.
* Altura de la capa límite planetaria.
* Índice de vegetación de diferencia normalizada, NDVI.
* Índice de edificación de diferencia normalizada, NDBI.

De este modo, cada muestra de entrada de la red presenta una estructura tridimensional formada por el número de observaciones, la longitud de la ventana temporal definida mediante `input_size` y las once variables utilizadas como entrada, correspondientes a la concentración histórica de CO y las diez variables exógenas.

Para la construcción de las secuencias del conjunto de validación se utilizan además las últimas `input_size` observaciones del conjunto de entrenamiento como contexto temporal. De esta forma, la primera predicción correspondiente al periodo de validación puede aprovechar las observaciones inmediatamente anteriores del final del conjunto de entrenamiento, evitando perder las primeras horas del periodo de validación y reproduciendo de forma más realista el proceso de predicción temporal.

La arquitectura de las redes evaluadas está compuesta por una o dos capas LSTM seguidas de una capa densa de una única neurona, encargada de generar la predicción final de la concentración de CO. Cuando se utilizan dos capas LSTM, la primera devuelve la secuencia completa de estados mediante `return_sequences = True`, permitiendo que dicha información sea procesada posteriormente por la segunda capa. La última capa LSTM devuelve únicamente su salida final, que se conecta con la capa `Dense(1)` encargada de proporcionar la predicción.

Los principales hiperparámetros evaluados son:

* El número de unidades o neuronas de cada capa LSTM, mediante `units`, considerando 32 y 64 unidades.
* El número de capas recurrentes utilizadas en la arquitectura, mediante `num_layers`, evaluando modelos con una y dos capas LSTM.
* La proporción de unidades desactivadas aleatoriamente durante el entrenamiento, mediante `dropout`, considerando valores de 0 y 0,2.
* El algoritmo utilizado para actualizar los pesos de la red durante el entrenamiento, mediante `optimizer`, comparando los optimizadores Adam y RMSprop.
* El número de épocas de entrenamiento, mediante `epochs`.
* El tamaño de los lotes utilizados para actualizar los parámetros de la red, mediante `batch_size`, considerando tamaños de 32 y 64 observaciones.
* El número de observaciones temporales anteriores empleadas para realizar cada predicción, mediante `input_size`, considerando ventanas de 24, 48, 72 y 168 horas.

El hiperparámetro `units` controla la capacidad de representación de cada capa LSTM. Un mayor número de unidades permite a la red aprender relaciones temporales más complejas, aunque también incrementa el número de parámetros y el coste computacional del modelo.

Por su parte, `num_layers` permite comparar arquitecturas de diferente profundidad. Una única capa LSTM representa una arquitectura más sencilla, mientras que el empleo de dos capas recurrentes permite construir representaciones temporales jerárquicas de mayor complejidad.

El parámetro `dropout` actúa como mecanismo de regularización. Durante el entrenamiento, una determinada proporción de unidades se desactiva aleatoriamente, reduciendo la dependencia excesiva entre neuronas y tratando de limitar el sobreajuste sobre el conjunto de entrenamiento.

También se comparan los optimizadores Adam y RMSprop, ambos ampliamente empleados en el entrenamiento de redes neuronales recurrentes. Estos algoritmos determinan cómo se actualizan los pesos de la red a partir de los gradientes calculados durante el proceso de aprendizaje.

Por último, `epochs` determina el número de veces que la red procesa completamente el conjunto de entrenamiento, mientras que `batch_size` establece el número de secuencias utilizadas en cada actualización de los pesos.

Con el objetivo de garantizar la reproducibilidad de los experimentos, antes de entrenar cada configuración se fija la misma semilla aleatoria para NumPy, Python y TensorFlow. Además, se limpia el estado previo de Keras antes de construir cada nuevo modelo, evitando que las diferentes configuraciones evaluadas interfieran entre sí.

In [ ]:
param_grid = {

    # Número de neuronas de las capas LSTM
    "units": [32, 64],

    # Número de capas LSTM
    "num_layers": [1, 2],

    # Dropout
    "dropout": [0.0, 0.2],

    # Optimizador
    "optimizer": ["adam", "rmsprop"],

    # Número de épocas
    "epochs": [50, 100],

    # Tamaño del batch
    "batch_size": [32, 64],

    # Número de horas anteriores consideradas
    "input_size": [24, 48, 72, 168]
}

In [69]:
# ==============================================================================
# Generación de todas las combinaciones posibles
# ==============================================================================

configuraciones = list(
    ParameterGrid(param_grid)
)

print(
    f"Número total de configuraciones que se evaluarán: "
    f"{len(configuraciones)}"
)

Número total de configuraciones que se evaluarán: 256


In [70]:
# ==============================================================================
# Función para construir el modelo LSTM
# ==============================================================================

def construir_modelo_lstm(
    input_shape,
    units,
    num_layers,
    dropout,
    optimizer
):
    """
    Construye y compila una red LSTM.

    Parámetros
    ----------
    input_shape : tuple
        Dimensiones de cada secuencia:
        (input_size, número de variables).

    units : int
        Número de unidades de cada capa LSTM.

    num_layers : int
        Número de capas LSTM.

    dropout : float
        Proporción de dropout aplicada en las capas LSTM.

    optimizer : str
        Optimizador empleado durante el entrenamiento.

    Retorna
    -------
    model : tf.keras.Model
        Modelo LSTM compilado.
    """

    model = Sequential()

    # --------------------------------------------------------------------------
    # Primera capa LSTM
    # --------------------------------------------------------------------------

    if num_layers == 1:

        model.add(
            LSTM(
                units=units,
                dropout=dropout,
                return_sequences=False,
                input_shape=input_shape
            )
        )

    else:

        model.add(
            LSTM(
                units=units,
                dropout=dropout,
                return_sequences=True,
                input_shape=input_shape
            )
        )

        # ----------------------------------------------------------------------
        # Capas LSTM adicionales
        # ----------------------------------------------------------------------

        for i in range(num_layers - 1):

            ultima_capa = (i == num_layers - 2)

            model.add(
                LSTM(
                    units=units,
                    dropout=dropout,
                    return_sequences=not ultima_capa
                )
            )

    # --------------------------------------------------------------------------
    # Capa de salida
    # --------------------------------------------------------------------------

    model.add(
        Dense(1)
    )

    # --------------------------------------------------------------------------
    # Compilación
    # --------------------------------------------------------------------------

    model.compile(
        optimizer=optimizer,
        loss="mean_squared_error"
    )

    return model

In [71]:
# ==============================================================================
# Búsqueda de configuraciones LSTM
# ==============================================================================

def busqueda_configuraciones_lstm(
    train,
    validation,
    variables_exogenas,
    param_grid,
    scaler_y,
    seed=42
):
    """
    Realiza una búsqueda de hiperparámetros para un modelo LSTM
    utilizando ParameterGrid.

    La selección se realiza según el MSE obtenido sobre
    el conjunto de validación.

    Parámetros
    ----------
    train : pd.DataFrame
        Conjunto de entrenamiento escalado.

    validation : pd.DataFrame
        Conjunto de validación escalado.

    variables_exogenas : list
        Lista de variables exógenas utilizadas.

    param_grid : dict
        Diccionario con los hiperparámetros a evaluar.

    scaler_y : MinMaxScaler
        Escalador empleado para la variable objetivo.

    seed : int
        Semilla para garantizar reproducibilidad.

    Retorna
    -------
    resultados : pd.DataFrame
        Resultados obtenidos para todas las configuraciones.

    mejor_configuracion : dict
        Configuración que obtiene el menor MSE de validación.
    """

    # ==========================================================================
    # Generación de configuraciones
    # ==========================================================================

    configuraciones = list(ParameterGrid(param_grid))

    resultados = []

    mejor_mse = np.inf
    mejor_configuracion = None

    # ==========================================================================
    # Evaluación de cada configuración
    # ==========================================================================

    for params in configuraciones:

        # ----------------------------------------------------------------------
        # Semillas
        # ----------------------------------------------------------------------

        np.random.seed(seed)
        random.seed(seed)
        tf.random.set_seed(seed)

        # Limpiar modelos anteriores de Keras
        tf.keras.backend.clear_session()

        # ----------------------------------------------------------------------
        # Creación de secuencias
        # ----------------------------------------------------------------------

        X_train, y_train = crear_dataset_lstm(
            train,
            input_size=params["input_size"],
            variables_exogenas=variables_exogenas
        )

        X_validation, y_validation = crear_dataset_lstm_validacion(
            train=train,
            validation=validation,
            input_size=params["input_size"],
            variables_exogenas=variables_exogenas
        )

        # ----------------------------------------------------------------------
        # Construcción del modelo
        # ----------------------------------------------------------------------

        model = construir_modelo_lstm(
            input_shape=(
                X_train.shape[1],
                X_train.shape[2]
            ),
            units=params["units"],
            num_layers=params["num_layers"],
            dropout=params["dropout"],
            optimizer=params["optimizer"]
        )

        # ----------------------------------------------------------------------
        # Entrenamiento
        # ----------------------------------------------------------------------

        model.fit(
            X_train,
            y_train,
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            verbose=0,
            shuffle=False
        )

        # ----------------------------------------------------------------------
        # Predicción sobre validación
        # ----------------------------------------------------------------------

        pred_validation = model.predict(
            X_validation,
            verbose=0
        )

        # ----------------------------------------------------------------------
        # Desescalado
        # ----------------------------------------------------------------------

        pred_validation_inv = scaler_y.inverse_transform(
            pred_validation
        ).ravel()

        y_validation_inv = scaler_y.inverse_transform(
            y_validation.reshape(-1, 1)
        ).ravel()

        # ----------------------------------------------------------------------
        # MSE de validación
        # ----------------------------------------------------------------------

        mse_validation = mean_squared_error(
            y_validation_inv,
            pred_validation_inv
        )

        # ----------------------------------------------------------------------
        # Guardar resultados
        # ----------------------------------------------------------------------

        resultado = params.copy()
        resultado["MSE_validacion"] = mse_validation

        resultados.append(resultado)

        # ----------------------------------------------------------------------
        # Actualización de la mejor configuración
        # ----------------------------------------------------------------------

        if mse_validation < mejor_mse:

            mejor_mse = mse_validation
            mejor_configuracion = params.copy()

        # ----------------------------------------------------------------------
        # Liberación de memoria
        # ----------------------------------------------------------------------

        del model
        del X_train
        del y_train
        del X_validation
        del y_validation

        tf.keras.backend.clear_session()

    # ==========================================================================
    # DataFrame final de resultados
    # ==========================================================================

    resultados = pd.DataFrame(resultados)

    resultados = resultados.sort_values(
        by="MSE_validacion",
        ascending=True
    ).reset_index(drop=True)

    # ==========================================================================
    # Impresión de la mejor configuración
    # ==========================================================================

    print("Mejores parámetros:\n")

    for parametro, valor in mejor_configuracion.items():
        print(f"{parametro}: {valor}")

    print(
        f"\nMSE de validación: "
        f"{mejor_mse:.6f}"
    )

    return resultados, mejor_configuracion

In [72]:
resultados_lstm, mejor_configuracion_lstm = busqueda_configuraciones_lstm(
    train=train_lstm,
    validation=validation_lstm,
    variables_exogenas=variables_exogenas,
    param_grid=param_grid,
    scaler_y=scaler_y,
    seed=42
)


Mejores parámetros:

units: 64
num_layers: 2
dropout: 0.2
optimizer: adam
epochs: 100
batch_size: 64
input_size: 72

MSE de validación: 0.024021


In [73]:
# Mejores parámetros obtenidos tras la búsqueda de hiperparámetros

mejores_parametros = {
    "units": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "optimizer": "adam",
    "epochs": 100,
    "batch_size": 64,
    "input_size": 72
}

In [ ]:
def ENTRENAR_EVALUAR_LSTM(
    train,
    validation,
    variables_exogenas,
    mejores_parametros,
    scaler_y,
    seed=42
):
    """
    Entrena el modelo LSTM con el conjunto de entrenamiento y evalúa 
    las predicciones sobre el conjunto de validación.

    Parámetros
    ----------
    train : pd.DataFrame
        Conjunto de entrenamiento escalado.

    validation : pd.DataFrame
        Conjunto de validación escalado.

    variables_exogenas : list
        Lista de variables exógenas utilizadas como predictores.

    mejores_parametros : dict
        Diccionario con los mejores hiperparámetros encontrados.

    scaler_y : MinMaxScaler
        Escalador utilizado para la variable objetivo.

    seed : int, default=42
        Semilla utilizada para garantizar reproducibilidad.

    Retorna
    -------
    modelo : tf.keras.Model
        Modelo LSTM entrenado.

    metricas : dict
        Diccionario con las métricas obtenidas sobre validación.

    y_val_real : np.ndarray
        Valores reales de la variable objetivo sin escalar (validación).

    y_val_predicho : np.ndarray
        Predicciones del modelo sin escalar (validación).

    fechas_val : pd.Series
        Fechas correspondientes a las predicciones realizadas.
    """

    # ==========================================================================
    # Semillas para reproducibilidad
    # ==========================================================================

    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

    tf.keras.backend.clear_session()


    # ==========================================================================
    # Extracción de los mejores hiperparámetros
    # ==========================================================================

    units = mejores_parametros["units"]
    num_layers = mejores_parametros["num_layers"]
    dropout = mejores_parametros["dropout"]
    optimizer = mejores_parametros["optimizer"]
    epochs = mejores_parametros["epochs"]
    batch_size = mejores_parametros["batch_size"]
    input_size = mejores_parametros["input_size"]


    # ==========================================================================
    # Creación de las secuencias de entrenamiento
    # ==========================================================================

    X_train, y_train = CREAR_DATASET_LSTM(
        df=train,
        input_size=input_size,
        variables_exogenas=variables_exogenas
    )


    # ==========================================================================
    # Creación de las secuencias de validación
    # ==========================================================================

    X_val, y_val = CREAR_DATASET_LSTM_VALIDATION(
        train=train,
        validation=validation,
        input_size=input_size,
        variables_exogenas=variables_exogenas
    )


    # ==========================================================================
    # Construcción del modelo
    # ==========================================================================

    modelo = CONSTRUIR_MODELO_LSTM(
        input_shape=(
            X_train.shape[1],
            X_train.shape[2]
        ),
        units=units,
        num_layers=num_layers,
        dropout=dropout,
        optimizer=optimizer
    )


    # ==========================================================================
    # Entrenamiento del modelo
    # ==========================================================================

    modelo.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        shuffle=False
    )


    # ==========================================================================
    # Predicción sobre el conjunto de validación
    # ==========================================================================

    y_val_predicho = modelo.predict(
        X_val,
        verbose=0
    )


    # ==========================================================================
    # Desescalado
    # ==========================================================================

    y_val_real = scaler_y.inverse_transform(
        y_val.reshape(-1, 1)
    ).ravel()

    y_val_predicho = scaler_y.inverse_transform(
        y_val_predicho
    ).ravel()


    # ==========================================================================
    # Evaluación del modelo
    # ==========================================================================

    metricas = EVALUAR_METRICAS(
        y_real=y_val_real,
        y_predicho=y_val_predicho,
        num_parametros=len(variables_exogenas)
    )


    # ==========================================================================
    # Fechas correspondientes a las predicciones
    # ==========================================================================

    fechas_val = validation["ds"].reset_index(drop=True)


    # ==========================================================================
    # Retorno de resultados
    # ==========================================================================

    return (
        modelo,
        metricas,
        y_val_real,
        y_val_predicho,
        fechas_val
    )

In [ ]:

modelo_lstm_final, metricas_lstm, y_val_real, y_val_predicho, fechas_val = (
    ENTRENAR_EVALUAR_LSTM(
        train=train_lstm,
        validation=validation_lstm,
        variables_exogenas=variables_exogenas,
        mejores_parametros=mejores_parametros,
        scaler_y=scaler_y,
        seed=42
    )
)

Resultados de la evaluación del modelo
--------------------------------------
Error absoluto medio (MAE): 0.149463
Error cuadrático medio (MSE): 0.024021
Raíz del error cuadrático medio (RMSE): 0.154987
Error porcentual absoluto medio (MAPE): 21.75 %
Raíz del error cuadrático medio normalizada (NRMSE): 45.14 %
